In [3]:
import pandas as pd
import numpy as np
import pickle
import ast
import re
import emoji
from catboost import CatBoostRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# ---------------------------------------------------------
# 1. INITIALISATION & FONCTIONS (Issu de ton notebook)
# ---------------------------------------------------------
analyzer = SentimentIntensityAnalyzer()

def engineer_features(df, is_train=True, train_df=None):
    df = df.copy()
    
    # Uniformisation stricte de la colonne ID
    if 'id' in df.columns:
        df = df.rename(columns={'id': 'ID'})
    df['ID'] = df['ID'].astype(str).str.strip()
        
    df['description'] = df['description'].fillna('')
    df['desc_len'] = df['description'].apply(len)
    df['word_count'] = df['description'].apply(lambda x: len(str(x).split()))
    df['hashtag_count'] = df['description'].apply(lambda x: len(re.findall(r'#\w+', str(x))))
    df['mention_count'] = df['description'].apply(lambda x: len(re.findall(r'@\w+', str(x))))
    df['emoji_count'] = df['description'].apply(lambda x: sum(1 for char in str(x) if char in emoji.EMOJI_DATA))

    ski_slang_map = {"sick": "amazing", "gnarly": "impressive", "insane": "incredible", "crushing": "performing well", "send it": "go for it", "fail": "funny mistake"}
    def get_sentiment(text):
        text = str(text).lower()
        for word, replacement in ski_slang_map.items():
            text = text.replace(word, replacement)
        return analyzer.polarity_scores(text)['compound']

    df['sent_compound'] = df['description'].apply(get_sentiment)

    df['video_age'] = 2026 - df['release_year'].fillna(df['release_year'].median())
    
    # Extraction temporelle depuis l'ID (sécurisé avec try/except au cas où l'ID n'est pas purement numérique)
    try:
        df['post_date'] = df['ID'].astype(int).apply(lambda x: pd.Timestamp((int(x) >> 32), unit='s'))
        df['post_month'] = df['post_date'].dt.month
    except:
        df['post_month'] = 6

    df['artist_count'] = df['artists'].apply(lambda x: len(ast.literal_eval(x)) if pd.notna(x) and str(x).startswith('[') else 1)
    
    if 'video_duration' in df.columns:
        df['is_optimal_duration'] = df['video_duration'].between(6, 12).astype(int)
    else:
        df['is_optimal_duration'] = 0
        
    df['is_minimalist'] = (df['desc_len'] <= 20).astype(int)
    df['is_val_thorens'] = (df['channel'].str.contains('Val Thorens', na=False)).astype(int)

    # Autorité de la chaîne lissée
    if is_train and 'popularity' in df.columns:
        alpha = 5
        global_mean = np.log1p(df['popularity']).mean()
        agg = df.groupby('channel')['popularity'].transform(lambda x: (np.log1p(x).sum() + alpha * global_mean) / (len(x) + alpha))
        df['channel_authority'] = agg
    elif not is_train and train_df is not None:
        channel_authority_map = train_df.groupby('channel')['channel_authority'].mean()
        df['channel_authority'] = df['channel'].map(channel_authority_map).fillna(train_df['channel_authority'].mean())
    else:
        df['channel_authority'] = 0

    return df

# ---------------------------------------------------------
# 2. CHARGEMENT TEXTE/CSV DE BASE
# ---------------------------------------------------------
# ---------------------------------------------------------
# 2. CHARGEMENT TEXTE/CSV DE BASE
# ---------------------------------------------------------
print("Chargement des données texte (train/test)...")

# Ajout de skipinitialspace=True pour aider un peu le parser
X_train = pd.read_csv('X_train.csv', sep=';', skipinitialspace=True)
y_train = pd.read_csv('y_train.csv', sep=';', skipinitialspace=True)
X_test = pd.read_csv('X_test.csv', sep=';', skipinitialspace=True)

# 1. Le grand nettoyage : on enlève tous les espaces invisibles des noms de colonnes
X_train.columns = X_train.columns.str.strip()
y_train.columns = y_train.columns.str.strip()
X_test.columns = X_test.columns.str.strip()

# 2. Uniformisation : on force tout le monde à s'appeler 'ID' (en majuscule)
if 'id' in X_train.columns: X_train.rename(columns={'id': 'ID'}, inplace=True)
if 'id' in y_train.columns: y_train.rename(columns={'id': 'ID'}, inplace=True)
if 'id' in X_test.columns: X_test.rename(columns={'id': 'ID'}, inplace=True)

# 3. On nettoie aussi le contenu des IDs (on enlève les espaces et on met en string)
X_train['ID'] = X_train['ID'].astype(str).str.strip()
y_train['ID'] = y_train['ID'].astype(str).str.strip()
X_test['ID'] = X_test['ID'].astype(str).str.strip()

# 4. Le merge devient super simple, plus d'erreur !
train_data = X_train.merge(y_train, on='ID', how='inner')

# Application du feature engineering
train_data = engineer_features(train_data, is_train=True)
test_data = engineer_features(X_test, is_train=False, train_df=train_data)

# TF-IDF (Appliqué sur les mots les plus importants)
print("Vectorisation TF-IDF...")
tfidf = TfidfVectorizer(max_features=100, stop_words='english', ngram_range=(1,2))
tfidf_train = pd.DataFrame(tfidf.fit_transform(train_data['description']).toarray(), columns=tfidf.get_feature_names_out(), index=train_data.index)
tfidf_test = pd.DataFrame(tfidf.transform(test_data['description']).toarray(), columns=tfidf.get_feature_names_out(), index=test_data.index)

train_data = pd.concat([train_data, tfidf_train], axis=1)
test_data = pd.concat([test_data, tfidf_test], axis=1)

# ---------------------------------------------------------
# 3. CHARGEMENT ET FUSION DES MODALITÉS (Audio, Visuel, Embeddings)
# ---------------------------------------------------------
print("Chargement des features multimodales...")
df_audio = pd.read_csv('audio_features_ready.csv')
df_visual = pd.read_csv('features_visual2.csv')
df_visual_add = pd.read_csv('visual_features_decord.csv')
df_objects = pd.read_csv('object_detection_features.csv')

with open('embeddings_video.pkl', 'rb') as f:
    embeddings_dict = pickle.load(f)

df_emb = pd.DataFrame.from_dict(embeddings_dict, orient='index')
df_emb.index.name = 'ID'
df_emb.columns = [f'emb_{i}' for i in range(df_emb.shape[1])]
df_emb.reset_index(inplace=True)

# 1. Nettoyage de df_audio et df_visual (Classique)
for df in [df_audio, df_visual, df_visual_add]:
    df.columns = df.columns.astype(str).str.strip()
    id_col = [c for c in df.columns if c.lower() == 'id']
    if id_col:
        df.rename(columns={id_col[0]: 'ID'}, inplace=True)
    df['ID'] = df['ID'].astype(str).str.strip()

# 2. NETTOYAGE SPÉCIFIQUE POUR LES OBJETS (Le fameux 'VIDEO_xxx')
df_objects.columns = df_objects.columns.astype(str).str.strip()
if 'video_id' in df_objects.columns:
    # On retire le préfixe 'VIDEO_' pour ne garder que l'ID numérique
    df_objects['ID'] = df_objects['video_id'].astype(str).str.replace('VIDEO_', '', regex=False)
    df_objects.drop(columns=['video_id'], inplace=True) # On nettoie l'ancienne colonne
else:
    # Si la colonne s'appelle autrement, on prend la 1ère colonne par défaut
    df_objects.rename(columns={df_objects.columns[0]: 'ID'}, inplace=True)
    df_objects['ID'] = df_objects['ID'].astype(str).str.replace('VIDEO_', '', regex=False)

# Nettoyage final des espaces
df_objects['ID'] = df_objects['ID'].astype(str).str.strip()
df_emb['ID'] = df_emb['ID'].astype(str).str.strip()
# Fonction de fusion robuste à gauche (pour ne pas perdre de lignes Test)
def merge_multimodal(df_base):
    res = df_base.merge(df_audio, on='ID', how='left')
    res = res.merge(df_visual, on='ID', how='left')
    res = res.merge(df_visual_add, on='ID', how='left')
    res = res.merge(df_objects, on='ID', how='left')
    res = res.merge(df_emb, on='ID', how='left')
    return res

print("Fusion de toutes les données multimodales...")
train_full = merge_multimodal(train_data)
test_full = merge_multimodal(test_data)

# ---------------------------------------------------------
# 4. PRÉPARATION CATBOOST (LA VERSION INFAILLIBLE)
# ---------------------------------------------------------
print("Préparation des données pour CatBoost...")
y_train_log = np.log1p(train_full['popularity'])

# On supprime les colonnes inutiles, redondantes ou trop uniques
cols_to_drop = ['ID', 'popularity', 'description', 'artists', 'release_year', 'post_date', 
                'filepath', 'download_timing', 'uploader', 'uploader_short', 'vid', 'uid']
features = [c for c in train_full.columns if c not in cols_to_drop]

X_train_final = train_full[features].copy()
X_test_final = test_full[features].copy()

# DÉTECTION AUTOMATIQUE DES CATÉGORIES
# On récupère toutes les colonnes qui sont de type 'object' ou 'string' (texte)
cat_features = X_train_final.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"📌 Colonnes catégorielles détectées et passées à CatBoost : {cat_features}")

# Nettoyage express de toutes les colonnes texte trouvées
for col in cat_features:
    # On remplace les trous par 'Unknown', on force en texte, et on enlève les espaces en trop
    X_train_final[col] = X_train_final[col].fillna('Unknown').astype(str).str.strip()
    X_test_final[col] = X_test_final[col].fillna('Unknown').astype(str).str.strip()

# ---------------------------------------------------------
# 5. ENTRAÎNEMENT & PRÉDICTION
# ---------------------------------------------------------
print(f"Entraînement du modèle CatBoost Multimodal ({X_train_final.shape[1]} features)...")
model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    cat_features=cat_features, # CatBoost sait maintenant exactement qui est du texte !
    random_seed=42,
    verbose=100
)

# Entraînement final
model.fit(X_train_final, y_train_log)

print("Génération des prédictions Test...")
preds_log = model.predict(X_test_final)

# On repasse du Log à la valeur normale
preds = np.expm1(preds_log)
preds = np.clip(preds, 0, None) # Pas de popularité négative

submission = pd.DataFrame({
    'ID': test_full['ID'],
    'popularity': preds
})

submission.to_csv('submission_catboost_multimodal.csv', index=False)
print(f"✅ Fichier 'submission_catboost_multimodal.csv' généré ! Nombre de lignes : {len(submission)}")

Chargement des données texte (train/test)...
Vectorisation TF-IDF...
Chargement des features multimodales...
Fusion de toutes les données multimodales...
Préparation des données pour CatBoost...
📌 Colonnes catégorielles détectées et passées à CatBoost : ['album', 'artist', 'channel', 'format', 'track']
Entraînement du modèle CatBoost Multimodal (2741 features)...
0:	learn: 0.1844633	total: 1s	remaining: 25m 4s
100:	learn: 0.1062387	total: 22.8s	remaining: 5m 16s
200:	learn: 0.0773533	total: 44.8s	remaining: 4m 49s
300:	learn: 0.0539457	total: 1m 5s	remaining: 4m 21s
400:	learn: 0.0379284	total: 1m 26s	remaining: 3m 56s
500:	learn: 0.0269451	total: 1m 46s	remaining: 3m 33s
600:	learn: 0.0195393	total: 2m 6s	remaining: 3m 9s
700:	learn: 0.0144004	total: 2m 28s	remaining: 2m 48s
800:	learn: 0.0106568	total: 2m 48s	remaining: 2m 27s
900:	learn: 0.0081351	total: 3m 10s	remaining: 2m 6s
1000:	learn: 0.0063791	total: 3m 31s	remaining: 1m 45s
1100:	learn: 0.0052759	total: 3m 51s	remaining: 1m 